# CÁC MÔ HÌNH PHI THAM SỐ — sổ tay thực hành (bản gọn, không kèm hình)

Notebook này minh họa **mọi mục chính (1–8)** của tài liệu *Các mô hình phi tham số* bằng mã **numpy viết từ đầu**
(scikit-learn chỉ dùng để đối chiếu và để lấy dữ liệu chữ số). Cách dùng:

- Chạy các ô **từ trên xuống dưới** (các ô sau dùng hàm đã định nghĩa ở ô trước).
- Các ô có chú thích `GIẢI THÍCH:` trong mã hoặc ô *Markdown* ngay sau đó giải thích những đoạn khó.
- File được lưu **không kèm hình** cho gọn: các ô vẽ hình ở cuối sẽ hiện hình khi bạn chạy trong Jupyter.
- Dữ liệu mô phỏng sinh bằng numpy với hạt giống cố định, nên con số có thể lệch vài phần trăm so với các bảng trong tài liệu
  (vốn dùng scikit-learn để chia tập); kết luận định tính giữ nguyên.

**Ký hiệu** (giống tài liệu): $\mathbf{X}\in\mathbb{R}^{n\times d}$ ma trận đặc trưng, $\mathbf{Y}\in\mathbb{R}^{n\times K}$ ma trận one-hot,
$\mathbf{X}_0\in\mathbb{R}^{m\times d}$ các điểm truy vấn. Tên biến dùng tiếng Việt không dấu.

In [1]:
import numpy as np
from math import gamma, pi
np.set_printoptions(precision=3, suppress=True)

def mot_nong(y, K=None):
    K = int(y.max()) + 1 if K is None else K
    return np.eye(K)[y]                       # hàng i của ma trận đơn vị = vector one-hot của nhãn y_i

def logsumexp(S):
    # GIẢI THÍCH: log(sum(exp(S))) theo từng hàng nhưng ổn định số. Trừ giá trị lớn nhất m trước khi lấy exp
    # (exp(S - m) <= 1 nên không tràn số), rồi cộng m lại: log(sum(exp(S))) = m + log(sum(exp(S - m))).
    m = S.max(axis=1, keepdims=True)
    return m[:, 0] + np.log(np.exp(S - m).sum(axis=1))

def softmax(S):
    return np.exp(S - logsumexp(S)[:, None])

class ChuanHoa:
    # Chuẩn hóa z-score. CHỈ học (mu, sigma) từ tập huấn luyện rồi áp dụng cho mọi tập (tránh rò thông tin).
    def fit(self, X):
        self.mu, self.sigma = X.mean(axis=0), X.std(axis=0) + 1e-12
        return self
    def transform(self, X):
        return (X - self.mu) / self.sigma

def chia_phan_tang(y, ti_le_test=0.3, seed=0):
    # Chia train/test sao cho mỗi lớp giữ cùng tỉ lệ.
    rng = np.random.default_rng(seed); tr, te = [], []
    for c in np.unique(y):
        idx = rng.permutation(np.where(y == c)[0]); n_te = int(round(ti_le_test * len(idx)))
        te += list(idx[:n_te]); tr += list(idx[n_te:])
    return np.array(tr), np.array(te)

In [2]:
def tao_du_lieu_sinh_vien(n_moi_lop=200, seed=0):
    """3 lớp 'rớt/đạt/giỏi', 2 đặc trưng (giờ tự học, số buổi vắng); đám Gauss chồng lấn."""
    rng = np.random.default_rng(seed)
    mu = [(3.5, 4.2), (6.5, 2.4), (9.5, 1.0)]
    sd = [(1.6, 1.3), (1.7, 1.2), (1.6, 1.0)]
    X, y = [], []
    for c in range(3):
        X.append(np.c_[rng.normal(mu[c][0], sd[c][0], n_moi_lop),
                       rng.normal(mu[c][1], sd[c][1], n_moi_lop)])
        y += [c] * n_moi_lop
    X = np.vstack(X)
    X[:, 0] = np.clip(np.round(X[:, 0], 1), 0, None)   # giờ học >= 0, làm tròn 0,1
    X[:, 1] = np.clip(np.round(X[:, 1]), 0, None)      # số buổi vắng là số nguyên >= 0
    return X, np.array(y)

def tao_ba_vong_tron(n_moi_lop=200, seed=0):
    """Lớp 0 là hình tròn đặc bán kính 1; lớp 1, 2 là hai vòng bán kính 3 và 5 (mục 1.1)."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for k, r in enumerate([1, 3, 5]):
        theta = rng.uniform(0, 2 * np.pi, n_moi_lop)
        # GIẢI THÍCH: lớp 0 lấy bán kính = căn(U) để điểm phân bố ĐỀU trên hình tròn đặc
        # (nếu lấy bán kính U thì điểm dồn về tâm). Hai vòng còn lại: bán kính r + nhiễu Gauss.
        rr = np.sqrt(rng.uniform(0, 1, n_moi_lop)) if k == 0 else r + rng.normal(0, 0.3, n_moi_lop)
        X.append(np.c_[rr * np.cos(theta), rr * np.sin(theta)])
        y += [k] * n_moi_lop
    return np.vstack(X), np.array(y)

def ham_that_sin(x):
    return np.sin(1.2 * x) + 0.15 * x

def tao_du_lieu_sin(n=100, sigma=0.3, seed=0):
    """Bài toán hồi quy y = sin(1,2x) + 0,15x + nhiễu, x ~ U(0,10) (mục 3)."""
    rng = np.random.default_rng(seed)
    x = np.sort(rng.uniform(0, 10, n))
    return x, ham_that_sin(x) + rng.normal(0, sigma, n)

def mat_do_that_hai_dinh(x):
    """Mật độ thật của hỗn hợp 0,6 N(-1,5; 0,7^2) + 0,4 N(2; 1^2) (mục 4)."""
    def gauss(t, mu, sd):
        return np.exp(-0.5 * ((t - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))
    return 0.6 * gauss(x, -1.5, 0.7) + 0.4 * gauss(x, 2.0, 1.0)

def lay_mau_hai_dinh(n, seed=0):
    rng = np.random.default_rng(seed)
    z = rng.random(n) < 0.6
    return np.where(z, rng.normal(-1.5, 0.7, n), rng.normal(2.0, 1.0, n))

## Mục 1–2. Tham số hay phi tham số? Từ Softmax đến $k$ láng giềng gần nhất

**Mô hình tham số** chọn trước dạng hàm (Softmax: $\hat{\mathbf P}=\mathrm{softmax}(\tilde{\mathbf X}\mathbf W)$) rồi tìm $\mathbf W$ bằng tối ưu hóa.
**k-NN** không có bước tối ưu: dự đoán là trung bình nhãn của $k$ láng giềng,
$$\hat{\mathbf P}=\tfrac1k\,\mathbf N\mathbf Y,\qquad N_{ji}=1\ \text{nếu mẫu } i \text{ thuộc } k \text{ láng giềng của điểm truy vấn } j.$$
Ô dưới huấn luyện Softmax bằng Gradient Descent rồi so với k-NN trên **ba vòng tròn đồng tâm** (dạng hàm tuyến tính sai).

In [3]:
class SoftmaxHoiQuy:
    """Softmax Regression huấn luyện bằng Gradient Descent (đại diện mô hình THAM SỐ)."""

    def __init__(self, toc_do_hoc=0.5, so_vong=1500, lam=1e-3):
        self.lr, self.T, self.lam = toc_do_hoc, so_vong, lam

    def fit(self, X, Y):
        self.cn = ChuanHoa().fit(X)
        Xt = np.c_[np.ones(len(X)), self.cn.transform(X)]            # thêm cột 1 (hệ số chặn)
        self.W = np.zeros((Xt.shape[1], Y.shape[1]))
        for _ in range(self.T):
            P = softmax(Xt @ self.W)
            # GIẢI THÍCH: với cross-entropy + softmax, gradient theo W gọn như hồi quy tuyến
            # tính: (1/n) Xt^T (P - Y). Hệ số chặn (hàng 0 của W) không bị điều chuẩn.
            grad = Xt.T @ (P - Y) / len(Xt)
            grad[1:] += self.lam * self.W[1:]
            self.W -= self.lr * grad
        return self

    def xac_suat(self, X0):
        return softmax(np.c_[np.ones(len(X0)), self.cn.transform(X0)] @ self.W)

    def du_doan(self, X0):
        return self.xac_suat(X0).argmax(axis=1)


X, y = tao_ba_vong_tron(200, seed=0)
tr, te = chia_phan_tang(y, 0.3, seed=0)
sm = SoftmaxHoiQuy().fit(X[tr], mot_nong(y[tr]))
print(f"Softmax trên (x1, x2): train {np.mean(sm.du_doan(X[tr]) == y[tr]):.3f} | test {np.mean(sm.du_doan(X[te]) == y[te]):.3f}")

Softmax trên (x1, x2): train 0.410 | test 0.400


**Giải thích mã khó 1 — ma trận khoảng cách không vòng lặp.** Với $\lVert\mathbf u-\mathbf v\rVert^2=\lVert\mathbf u\rVert^2-2\,\mathbf u\mathbf v^\top+\lVert\mathbf v\rVert^2$ ta có
$$\mathbf D^{(2)}=\mathbf a\,\mathbf 1_n^\top-2\,\mathbf X_0\mathbf X^\top+\mathbf 1_m\,\mathbf b^\top,\quad a_j=\lVert\mathbf x_{0,j}\rVert^2,\ b_i=\lVert\mathbf x_i\rVert^2 .$$
Trong numpy: `a` có hình $(m,1)$ và `b` có hình $(1,n)$ nên `a - 2*X0@X.T + b` tự "lặp" (broadcasting) thành ma trận $m\times n$.
Sai số làm tròn có thể cho số âm cỡ $-10^{-16}$, nên cắt về $0$ trước khi lấy căn.

**Giải thích mã khó 2 — dựng ma trận láng giềng $\mathbf N$.** `N[hang, idx] = ...` với `hang` hình $(m,1)$ và `idx` hình $(m,k)$:
hai chỉ số được broadcast thành $(m,k)$ nên mỗi hàng $j$ có đúng $k$ ô `N[j, idx[j, t]]` được gán. `argsort(kind="stable")` cho thứ tự lặp lại được khi có hai điểm cách đều.

In [4]:
def binh_phuong_khoang_cach(X0, X):
    """Ma trận D2 (m x n) với D2[j, i] = ||X0[j] - X[i]||^2, không dùng vòng lặp.

    GIẢI THÍCH: khai triển ||u - v||^2 = ||u||^2 - 2 u.v + ||v||^2. Ba thành phần gom thành
    một biểu thức ma trận: a (m x 1) cộng b (1 x n) nhờ "broadcasting" của numpy (cột a được
    lặp ngang, hàng b được lặp dọc), trừ 2 * X0 @ X.T (phép nhân ma trận m x d với d x n,
    phần nặng nhất nhưng thư viện đại số tuyến tính làm rất nhanh).
    """
    a = np.sum(X0 ** 2, axis=1, keepdims=True)    # (m, 1)
    b = np.sum(X ** 2, axis=1)[None, :]           # (1, n)
    D2 = a - 2.0 * X0 @ X.T + b
    return np.maximum(D2, 0.0)                    # sai số làm tròn có thể cho số âm ~ -1e-16

def knn_xac_suat(X0, X, Y, k=3, co_trong_so=False, eps=1e-12):
    """Ma trận xác suất lớp P_hat (m x K) = (1/k) N Y  (hoặc N_w Y nếu có trọng số)."""
    m, n = X0.shape[0], X.shape[0]
    D2 = binh_phuong_khoang_cach(X0, X)
    # GIẢI THÍCH: argsort(stable) cho thứ tự lặp lại được khi có hai điểm cách đều (hòa).
    # Có thể thay bằng np.argpartition(D2, k-1)[:, :k] (O(n) thay vì O(n log n)) nhưng khi
    # hòa thì chọn láng giềng nào là tùy ý.
    idx = np.argsort(D2, axis=1, kind="stable")[:, :k]       # (m, k) chỉ số k láng giềng
    N = np.zeros((m, n))
    hang = np.arange(m)[:, None]                              # (m, 1)
    # GIẢI THÍCH: "fancy indexing" - cặp (hang, idx) có hình (m,1) và (m,k) được broadcast
    # thành (m,k): ô N[j, idx[j, t]] được gán, tức đánh dấu k láng giềng của mỗi hàng.
    N[hang, idx] = 1.0 / (D2[hang, idx] + eps) if co_trong_so else 1.0
    return (N @ Y) / N.sum(axis=1, keepdims=True)             # chia tổng hàng = chuẩn hóa

def knn_du_doan(X0, X, y, k=3, co_trong_so=False):
    return knn_xac_suat(X0, X, mot_nong(y), k, co_trong_so).argmax(axis=1)


# Ví dụ tính tay (mục 2.6): 9 sinh viên (rớt=0, đạt=1, giỏi=2), 3 điểm truy vấn, k = 3
S9 = np.array([[2,4,0],[3,3,0],[4,5,0],[6,3,1],[7,2,1],[6,4,1],[9,1,2],[10,0,2],[8,0,2]], float)
X9, y9 = S9[:, :2], S9[:, 2].astype(int)
X0 = np.array([[5, 3], [8, 2], [3, 2]], float)
print("D^(2) =\n", binh_phuong_khoang_cach(X0, X9).astype(int))
print("P_hat, bỏ phiếu đa số:\n", knn_xac_suat(X0, X9, mot_nong(y9), 3))
print("P_hat, trọng số 1/d^2:\n", knn_xac_suat(X0, X9, mot_nong(y9), 3, co_trong_so=True))

# k-NN trên ba vòng tròn
cn = ChuanHoa().fit(X[tr]); Ztr, Zte = cn.transform(X[tr]), cn.transform(X[te])
for k in (1, 15):
    print(f"k-NN k={k}: test {np.mean(knn_du_doan(Zte, Ztr, y[tr], k) == y[te]):.3f}")

D^(2) =
 [[10  4  5  1  5  2 20 34 18]
 [40 26 25  5  1  8  2  8  4]
 [ 5  1 10 10 16 13 37 53 29]]
P_hat, bỏ phiếu đa số:
 [[0.333 0.667 0.   ]
 [0.    0.333 0.667]
 [1.    0.    0.   ]]
P_hat, trọng số 1/d^2:
 [[0.143 0.857 0.   ]
 [0.    0.571 0.429]
 [1.    0.    0.   ]]
k-NN k=1: test 0.994
k-NN k=15: test 1.000


**Chọn $k$ bằng kiểm định chéo và vai trò của chuẩn hóa (mục 2.4–2.5).** Quy tắc vàng: *mọi bước học từ dữ liệu* (kể cả chuẩn hóa) phải nằm **trong** vòng lặp kiểm định chéo,
nếu không thông tin của phần kiểm định "rò" vào bước huấn luyện. Ô dưới cũng cho thấy chỉ cần đổi đơn vị "giờ" thành "phút" là k-NN không chuẩn hóa kém đi.

In [5]:
def chon_k_kiem_dinh_cheo(X, y, ds_k, V=5, seed=0):
    """Chọn k bằng kiểm định chéo V phần. Chuẩn hóa được HỌC LẠI trong từng phần (tránh rò)."""
    rng = np.random.default_rng(seed)
    nhom = rng.permutation(len(y)) % V                        # gán mỗi mẫu vào một trong V phần
    diem = {k: [] for k in ds_k}
    for v in range(V):
        tr, va = nhom != v, nhom == v
        cn = ChuanHoa().fit(X[tr])                            # chỉ dùng V-1 phần để chuẩn hóa
        Ztr, Zva = cn.transform(X[tr]), cn.transform(X[va])
        for k in ds_k:
            diem[k].append((knn_du_doan(Zva, Ztr, y[tr], k) == y[va]).mean())
    cv = {k: float(np.mean(v)) for k, v in diem.items()}
    return max(cv, key=cv.get), cv


X, y = tao_du_lieu_sinh_vien(200, seed=0)
tr, te = chia_phan_tang(y, 0.3, seed=0)
k_tot, cv = chon_k_kiem_dinh_cheo(X[tr], y[tr], [1, 3, 5, 9, 15, 25, 50])
print("CV theo k:", {k: round(v, 3) for k, v in cv.items()}, "-> k tốt nhất =", k_tot)
cn = ChuanHoa().fit(X[tr])
acc_chuan = np.mean(knn_du_doan(cn.transform(X[te]), cn.transform(X[tr]), y[tr], k_tot) == y[te])
X_phut = X.copy(); X_phut[:, 0] *= 60                      # đổi đơn vị giờ -> phút
acc_tho = np.mean(knn_du_doan(X_phut[te], X_phut[tr], y[tr], k_tot) == y[te])
print(f"test có chuẩn hóa: {acc_chuan:.3f} | đổi giờ->phút, KHÔNG chuẩn hóa: {acc_tho:.3f}")
sm = SoftmaxHoiQuy().fit(X[tr], mot_nong(y[tr]))
print(f"Softmax (dạng hàm đúng với đám Gauss): test {np.mean(sm.du_doan(X[te]) == y[te]):.3f}")

CV theo k: {1: 0.719, 3: 0.767, 5: 0.802, 9: 0.802, 15: 0.812, 25: 0.807, 50: 0.798} -> k tốt nhất = 15
test có chuẩn hóa: 0.828 | đổi giờ->phút, KHÔNG chuẩn hóa: 0.750
Softmax (dạng hàm đúng với đám Gauss): test 0.828


## Mục 3. Hồi quy: $k$-NN, Nadaraya–Watson, LOESS

Tất cả đều có dạng *trung bình có trọng số* $\hat y_0=\sum_i s_i(\mathbf x_0)\,y_i$:

| phương pháp | trọng số $s_i$ |
|---|---|
| k-NN | $1/k$ cho $k$ láng giềng |
| Nadaraya–Watson | $\kappa(d_i/h)/\sum_l\kappa(d_l/h)$ |
| LOESS | khớp **đường thẳng** cục bộ: $\mathbf w(\mathbf x_0)=(\tilde{\mathbf X}^\top\mathbf V\tilde{\mathbf X})^{-1}\tilde{\mathbf X}^\top\mathbf V\mathbf y$, $\hat y_0=\tilde{\mathbf x}_0\mathbf w(\mathbf x_0)$ |

với $\mathbf V=\mathrm{diag}(v_i)$, $v_i=\kappa(d_i/h)$ — ma trận $\mathbf V$ **đổi theo điểm truy vấn**, nên LOESS giải một hệ phương trình nhỏ cho *mỗi* điểm.

In [6]:
def knn_hoi_quy(X0, X, y, k=3):
    D2 = binh_phuong_khoang_cach(X0, X)
    idx = np.argsort(D2, axis=1, kind="stable")[:, :k]
    return y[idx].mean(axis=1)                                # (1/k) N y

def ham_nhan(u, loai="gauss"):
    if loai == "gauss":
        return np.exp(-0.5 * u ** 2)
    if loai == "epanechnikov":
        return np.maximum(1.0 - u ** 2, 0.0)
    if loai == "deu":
        return (np.abs(u) <= 1.0).astype(float)
    raise ValueError(loai)

def nadaraya_watson(X0, X, y, h, loai="gauss"):
    G = ham_nhan(np.sqrt(binh_phuong_khoang_cach(X0, X)) / h, loai)   # (m, n)
    return (G @ y) / G.sum(axis=1)                            # = diag(G 1)^-1 G y

def loess(X0, X, y, h, loai="gauss", lam=1e-8):
    """Hồi quy tuyến tính trọng số địa phương: w(x0) = (X~^T V X~)^-1 X~^T V y."""
    Xt = np.c_[np.ones(len(X)), X]                            # X~: thêm cột 1
    Gv = ham_nhan(np.sqrt(binh_phuong_khoang_cach(X0, X)) / h, loai)  # (m, n): v_i(x0)
    ket_qua = np.empty(len(X0))
    for j in range(len(X0)):                                  # mỗi điểm truy vấn giải một hệ riêng
        v = Gv[j][:, None]                                    # (n, 1) = đường chéo của V
        # GIẢI THÍCH: không dựng ma trận đường chéo V (n x n) mà nhân từng hàng: (v * Xt)
        # bằng V @ Xt nhưng chỉ tốn O(n d). lam*I giúp ma trận không suy biến khi ít mẫu.
        A = Xt.T @ (v * Xt) + lam * np.eye(Xt.shape[1])
        w = np.linalg.solve(A, Xt.T @ (v[:, 0] * y))          # solve ổn định hơn inv(A) @ b
        ket_qua[j] = np.r_[1.0, X0[j]] @ w                    # x0~ w(x0)
    return ket_qua


# Ví dụ tính tay (mục 3.6): ở giữa (x0=3,7) và ở BIÊN (x0=6); dữ liệu gần tuyến tính
xs = np.array([1, 2, 3, 4, 5, 6.]); ys = np.array([4.0, 5.2, 5.8, 7.1, 8.0, 9.1]); Q = np.array([[3.7], [6.0]])
print("k-NN (k=3)      :", knn_hoi_quy(Q, xs[:, None], ys, 3))
print("Nadaraya-Watson :", nadaraya_watson(Q, xs[:, None], ys, 1.0))
print("LOESS           :", loess(Q, xs[:, None], ys, 1.0), "(giá trị quan sát tại x=6 là 9,1: hai phương pháp 'trung bình' bị thấp ở biên)")

k-NN (k=3)      : [6.967 8.067]
Nadaraya-Watson : [6.693 8.543]
LOESS           : [6.697 9.088] (giá trị quan sát tại x=6 là 9,1: hai phương pháp 'trung bình' bị thấp ở biên)


**Giải thích mã khó 3 — `loess`.** (i) `v * Xt` nhân từng hàng của $\tilde{\mathbf X}$ với $v_i$, tương đương $\mathbf V\tilde{\mathbf X}$ nhưng không dựng ma trận $n\times n$.
(ii) `np.linalg.solve(A, b)` giải $A\mathbf w=\mathbf b$ trực tiếp, ổn định hơn tính `inv(A) @ b`. (iii) `lam*I` nhỏ tránh ma trận suy biến khi lân cận ít mẫu.

**Giải thích mã khó 4 — bỏ-một-ra không cần huấn luyện lại.** Nếu $\hat{\mathbf y}=\mathbf S\mathbf y$ thì sai số khi bỏ mẫu $i$ là $(y_i-\hat y_i)/(1-S_{ii})$ — **chính xác**, không xấp xỉ.
Ô sau kiểm chứng bằng cách bỏ thật từng mẫu.

In [7]:
def ma_tran_lam_muot(x, h, kieu="nw"):
    """Ma trận làm trơn S (n x n) sao cho y_hat = S y trên chính các mẫu huấn luyện."""
    n = len(x)
    G = ham_nhan(np.abs(x[:, None] - x[None, :]) / h)
    if kieu == "nw":
        return G / G.sum(axis=1, keepdims=True)
    Xa = np.c_[np.ones(n), x]
    S = np.zeros((n, n))
    for j in range(n):                                        # hàng j của S = x_j~ (X~^T V X~)^-1 X~^T V
        v = G[j][:, None]
        A = np.linalg.inv(Xa.T @ (v * Xa) + 1e-8 * np.eye(2))
        S[j] = (np.array([1.0, x[j]]) @ A @ Xa.T) * G[j]
    return S

def loocv_bo_mot_ra(S, y):
    """Sai số bỏ-một-ra của bộ làm trơn tuyến tính mà KHÔNG phải huấn luyện lại n lần.

    GIẢI THÍCH: với y_hat = S y, dự đoán khi bỏ mẫu i là (y_hat_i - S_ii y_i)/(1 - S_ii),
    nên sai số bỏ-một-ra là (y_i - y_hat_i)/(1 - S_ii). Công thức này CHÍNH XÁC (không xấp xỉ)
    với bộ làm trơn tuyến tính có băng thông cố định (mục 3.4).
    """
    r = (y - S @ y) / (1.0 - np.diag(S))
    return float(np.mean(r ** 2))

def chon_k_hoi_quy_bo_mot_ra(x, y, ds_k):
    """Chọn k cho k-NN hồi quy một chiều bằng bỏ-một-ra (bỏ thật từng mẫu, vì k-NN không có
    công thức đóng như bộ làm trơn có băng thông cố định)."""
    D = np.abs(x[:, None] - x[None, :])
    np.fill_diagonal(D, np.inf)                       # GIẢI THÍCH: đặt khoảng cách tới chính mình = vô cực
    thu_tu = np.argsort(D, axis=1, kind="stable")     # -> mẫu i không bao giờ là láng giềng của chính nó
    loi = {k: np.mean((y - y[thu_tu[:, :k]].mean(axis=1)) ** 2) for k in ds_k}
    return min(loi, key=loi.get)


x, y = tao_du_lieu_sin(100, 0.3, seed=0)
xt = np.random.default_rng(1).uniform(0, 10, 400); yt = ham_that_sin(xt) + np.random.default_rng(2).normal(0, 0.3, 400)
X, Xt = x[:, None], xt[:, None]
rmse = lambda p: float(np.sqrt(np.mean((yt - p) ** 2)))

h0 = 0.6
bo_that = np.mean([(y[i] - nadaraya_watson(X[i:i+1], np.delete(X, i, 0), np.delete(y, i), h0)[0]) ** 2 for i in range(len(x))])
print(f"LOOCV (NW, h=0,6): công thức đóng {loocv_bo_mot_ra(ma_tran_lam_muot(x, h0), y):.6f} | bỏ thật từng mẫu {bo_that:.6f}")

hs = np.round(np.arange(0.1, 2.01, 0.05), 2)
h_nw = hs[int(np.argmin([loocv_bo_mot_ra(ma_tran_lam_muot(x, h, "nw"), y) for h in hs]))]
h_lo = hs[int(np.argmin([loocv_bo_mot_ra(ma_tran_lam_muot(x, h, "loess"), y) for h in hs]))]
k_tot = chon_k_hoi_quy_bo_mot_ra(x, y, range(1, 31))
print("RMSE kiểm tra (mức sàn do nhiễu = 0,300)")
print(f"  hồi quy tuyến tính : {rmse(np.polyval(np.polyfit(x, y, 1), xt)):.3f}")
print(f"  đa thức bậc 7      : {rmse(np.polyval(np.polyfit(x/10, y, 7), xt/10)):.3f}")
print(f"  k-NN (k={k_tot})        : {rmse(knn_hoi_quy(Xt, X, y, k_tot)):.3f}")
print(f"  Nadaraya-Watson (h={h_nw}): {rmse(nadaraya_watson(Xt, X, y, h_nw)):.3f}")
print(f"  LOESS (h={h_lo})          : {rmse(loess(Xt, X, y, h_lo)):.3f}")

LOOCV (NW, h=0,6): công thức đóng 0.116060 | bỏ thật từng mẫu 0.116060
RMSE kiểm tra (mức sàn do nhiễu = 0,300)
  hồi quy tuyến tính : 0.716
  đa thức bậc 7      : 0.305
  k-NN (k=5)        : 0.341
  Nadaraya-Watson (h=0.3): 0.325
  LOESS (h=0.3)          : 0.323


## Mục 4. Ước lượng mật độ phi tham số

Mọi phương pháp xuất phát từ $\hat f(\mathbf x)=\dfrac{k}{nV}$: **cố định $V$, đếm $k$** (histogram, KDE) hoặc **cố định $k$, đo $V$** (k láng giềng).
KDE đặt một "gò" Gauss trên mỗi mẫu: $\hat{\mathbf f}=\dfrac{1}{n(2\pi)^{d/2}h^d}\,\mathbf G\mathbf 1_n$, $G_{ji}=\exp\!\big(-D^{(2)}_{ji}/2h^2\big)$ — dùng lại ma trận khoảng cách của mục 2.

In [8]:
def histogram_mat_do(x0, x, delta, goc=0.0):
    """f(x) = n_j / (n * delta) với j là ô chứa x."""
    chi_so_x = np.floor((x - goc) / delta).astype(int)
    chi_so_0 = np.floor((np.atleast_1d(x0) - goc) / delta).astype(int)
    return np.array([(chi_so_x == b).sum() for b in chi_so_0]) / (len(x) * delta)

def kde(X0, X, h):
    """KDE nhân Gauss: f = (1/n) (2 pi)^(-d/2) h^(-d) * G 1_n, G[j,i] = exp(-D2[j,i]/(2h^2))."""
    n, d = X.shape
    G = np.exp(-binh_phuong_khoang_cach(X0, X) / (2.0 * h ** 2))
    return G.sum(axis=1) / (n * (2.0 * pi) ** (d / 2) * h ** d)

def the_tich_cau(d, r):
    return pi ** (d / 2) / gamma(d / 2 + 1) * r ** d          # omega_d * r^d

def knn_mat_do(X0, X, k):
    """f(x) = k / (n V_k(x)); V_k là thể tích hình cầu bán kính = khoảng cách tới láng giềng thứ k."""
    n, d = X.shape
    r = np.sqrt(np.sort(binh_phuong_khoang_cach(X0, X), axis=1)[:, k - 1])
    return k / (n * the_tich_cau(d, r))

def silverman(x):
    """Quy tắc Silverman: h = 0,9 min(sigma, IQR/1,34) n^(-1/5) (một chiều)."""
    q75, q25 = np.percentile(x, [75, 25])
    return 0.9 * min(x.std(ddof=1), (q75 - q25) / 1.34) * len(x) ** (-0.2)

def log_kha_nang_bo_mot_ra(X, h):
    """LCV(h) = (1/n) sum_i ln f_{-i}(x_i): mật độ tại x_i tính KHÔNG dùng chính x_i."""
    n, d = X.shape
    G = np.exp(-binh_phuong_khoang_cach(X, X) / (2.0 * h ** 2))
    # GIẢI THÍCH: đường chéo của G ứng với khoảng cách 0 (mẫu so với chính nó), giá trị 1.
    # Nếu giữ lại thì f(x_i) có thêm số hạng không đổi và LCV luôn tăng khi h -> 0 (học thuộc
    # cực đoan). Đặt đường chéo = 0 và chia cho (n - 1) chính là "bỏ một ra".
    np.fill_diagonal(G, 0.0)
    f = G.sum(axis=1) / ((n - 1) * (2.0 * pi) ** (d / 2) * h ** d)
    return float(np.mean(np.log(f + 1e-300)))


x = lay_mau_hai_dinh(300, 0); xt = lay_mau_hai_dinh(2000, 1); X, Xt = x[:, None], xt[:, None]
luoi = np.linspace(-5, 6, 1101); dl = luoi[1] - luoi[0]
ise = lambda f: float(np.sum((f - mat_do_that_hai_dinh(luoi)) ** 2) * dl)          # sai số bình phương tích phân
ll = lambda f_te: float(np.mean(np.log(np.maximum(f_te, 1e-6))))                    # log-hợp lý TB trên tập kiểm tra
print(f"Mật độ thật: log-hợp lý kiểm tra = {np.mean(np.log(mat_do_that_hai_dinh(xt))):.3f}")
mu, sd = x.mean(), x.std(); g = lambda t: np.exp(-0.5*((t-mu)/sd)**2) / (sd*np.sqrt(2*np.pi))
print(f"Gauss hợp lý cực đại : ISE {ise(g(luoi)):.4f} | log-hợp lý {ll(g(xt)):.3f}   (một đỉnh -> thiên lệch lớn)")
for delta in (0.1, 0.5, 2.0):
    print(f"Histogram Δ={delta:<3}      : ISE {ise(histogram_mat_do(luoi, x, delta)):.4f}")
h_s = silverman(x); hs = np.round(np.arange(0.05, 1.5, 0.01), 2)
h_l = hs[int(np.argmax([log_kha_nang_bo_mot_ra(X, h) for h in hs]))]
for ten, h in (("Silverman", h_s), ("hợp lý cực đại + KĐC", h_l)):
    print(f"KDE h={h:.3f} ({ten:<20}): ISE {ise(kde(luoi[:, None], X, h)):.4f} | log-hợp lý {ll(kde(Xt, X, h)):.3f}")
for k in (3, 15, 60):
    tp = np.sum(knn_mat_do(np.linspace(-200, 200, 200001)[:, None], X, k)) * 0.002
    print(f"k láng giềng k={k:<2}: ISE {ise(knn_mat_do(luoi[:, None], X, k)):.4f} | tích phân trên [-200,200] = {tp:.2f}  (>1: không phải mật độ hợp lệ)")

xs = np.array([1, 2, 2.5, 3, 7, 8, 8.5, 9.]); Q = np.array([[2.5], [5.0]])             # ví dụ tính tay mục 4.7
print("\nTính tay: histogram", histogram_mat_do(Q[:, 0], xs, 2.0), "| KDE h=1", kde(Q, xs[:, None], 1.0), "| kNN k=3", knn_mat_do(Q, xs[:, None], 3))

Mật độ thật: log-hợp lý kiểm tra = -1.835
Gauss hợp lý cực đại : ISE 0.0679 | log-hợp lý -2.077   (một đỉnh -> thiên lệch lớn)
Histogram Δ=0.1      : ISE 0.0432
Histogram Δ=0.5      : ISE 0.0084
Histogram Δ=2.0      : ISE 0.0409
KDE h=0.538 (Silverman           ): ISE 0.0093 | log-hợp lý -1.867
KDE h=0.400 (hợp lý cực đại + KĐC): ISE 0.0059 | log-hợp lý -1.855
k láng giềng k=3 : ISE 0.3199 | tích phân trên [-200,200] = 1.56  (>1: không phải mật độ hợp lệ)
k láng giềng k=15: ISE 0.0266 | tích phân trên [-200,200] = 1.37  (>1: không phải mật độ hợp lệ)
k láng giềng k=60: ISE 0.0119 | tích phân trên [-200,200] = 2.04  (>1: không phải mật độ hợp lệ)

Tính tay: histogram [0.188 0.   ] | KDE h=1 [0.154 0.017] | kNN k=3 [0.375 0.075]


**Giải thích mã khó 5 — vì sao `log_kha_nang_bo_mot_ra` phải đặt đường chéo bằng 0.** Đường chéo của $\mathbf G$ là khoảng cách của mẫu tới *chính nó* (giá trị $1$). Nếu giữ,
$\hat f(\mathbf x_i)$ luôn có thêm số hạng $\kappa_d(\mathbf 0)/(nh^d)\to\infty$ khi $h\to0$, nên log-hợp lý huấn luyện không bị chặn và luôn "chọn" $h\to0$ (học thuộc cực đoan).
Đặt đường chéo $=0$ và chia cho $n-1$ chính là $\hat f_{-i}(\mathbf x_i)$ — mật độ tại $\mathbf x_i$ tính **không dùng** $\mathbf x_i$.

## Mục 5. Bayes ngây thơ

Định lý Bayes: $P(c\mid\mathbf x)\propto\pi_c\prod_j p(x_j\mid c)$. "Huấn luyện" chỉ là các phép nhân ma trận với $\mathbf Y^\top$:
$$\hat{\boldsymbol\pi}=\tfrac1n\mathbf Y^\top\mathbf 1,\qquad \hat{\mathbf M}=\mathrm{diag}(\mathbf t)^{-1}\mathbf Y^\top\mathbf X,\qquad \mathbf C=\mathbf Y^\top\mathbf X\ \text{(ma trận đếm)}.$$
Dự đoán trong **không gian log** (tránh tràn số dưới) và hậu nghiệm $=\mathrm{softmax}$ của điểm số $s_c=\ln\pi_c+\sum_j\ln p(x_j\mid c)$.
Với Multinomial: $\mathbf S=\mathbf 1\ln\hat{\boldsymbol\pi}^\top+\mathbf X_0(\ln\hat{\boldsymbol\Theta})^\top$ — **cùng dạng** $\tilde{\mathbf X}\mathbf W$ của Softmax Regression, chỉ khác cách có $\mathbf W$.

In [9]:
class GaussNB:
    """Bayes ngây thơ Gauss. 'Huấn luyện' = vài phép nhân ma trận với Y^T (công thức đóng)."""
    def __init__(self, lam=1e-9):
        self.lam = lam

    def fit(self, X, Y):
        t = Y.sum(axis=0)                                     # t_c = số mẫu mỗi lớp
        self.pi = t / len(X)
        self.mu = (Y.T @ X) / t[:, None]                      # M = diag(t)^-1 Y^T X   (K x d)
        E2 = (Y.T @ (X * X)) / t[:, None]                     # trung bình của bình phương
        # phương sai = E[x^2] - (E[x])^2; cộng sàn nhỏ để không chia cho 0 (mục 7.4)
        self.var = E2 - self.mu ** 2 + self.lam * X.var(axis=0).max()
        return self

    def diem_so(self, X0):
        """s_c = ln pi_c + sum_j ln N(x_j; mu_cj, var_cj)."""
        S = np.log(self.pi)[None, :] + np.zeros((len(X0), 1))
        for c in range(len(self.pi)):
            S[:, c] += np.sum(-0.5 * np.log(2 * pi * self.var[c])
                              - 0.5 * (X0 - self.mu[c]) ** 2 / self.var[c], axis=1)
        return S

    def xac_suat(self, X0):
        return softmax(self.diem_so(X0))

    def du_doan(self, X0):
        return self.diem_so(X0).argmax(axis=1)

class MultiNB:
    """Bayes ngây thơ Multinomial (đếm từ) với làm trơn Laplace alpha."""
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, Y):
        self.pi = Y.sum(axis=0) / len(X)
        C = Y.T @ X                                           # ma trận đếm K x d
        self.theta = (C + self.alpha) / (C.sum(axis=1, keepdims=True) + self.alpha * X.shape[1])
        return self

    def diem_so(self, X0):
        # GIẢI THÍCH: S = 1 ln(pi)^T + X0 ln(Theta)^T  -  CÙNG DẠNG với Softmax Regression
        # S = X~ W, chỉ khác W được tính bằng đếm (hợp lý đồng thời) chứ không bằng GD (mục 5.5).
        return np.log(self.pi)[None, :] + X0 @ np.log(self.theta).T

    def xac_suat(self, X0):
        return softmax(self.diem_so(X0))

    def du_doan(self, X0):
        return self.diem_so(X0).argmax(axis=1)


def log_loss(y, P):
    P = np.clip(P, 1e-15, 1.0); return float(-np.mean(np.log(P[np.arange(len(y)), y])))

X, y = tao_du_lieu_sinh_vien(200, 0); tr, te = chia_phan_tang(y, 0.3, 0)
P = GaussNB().fit(X[tr], mot_nong(y[tr])).xac_suat(X[te])
print(f"Bayes ngây thơ Gauss: test {np.mean(P.argmax(1) == y[te]):.3f}, log-loss {log_loss(y[te], P):.3f}")

# đặc trưng nhiễu: Bayes ngây thơ chịu tốt hơn k-NN
rng = np.random.default_rng(0)
for q in (0, 20, 100):
    XX = np.c_[X, rng.normal(size=(len(X), q))]; cn = ChuanHoa().fit(XX[tr])
    a_nb = np.mean(GaussNB().fit(XX[tr], mot_nong(y[tr])).du_doan(XX[te]) == y[te])
    a_knn = np.mean(knn_du_doan(cn.transform(XX[te]), cn.transform(XX[tr]), y[tr], 18) == y[te])
    print(f"  q={q:<3} đặc trưng nhiễu: Bayes ngây thơ {a_nb:.3f} | k-NN {a_knn:.3f}")

# sao chép đặc trưng -> 'đếm bằng chứng nhiều lần' -> quá tự tin
for m in (1, 5, 10):
    A = np.c_[X, np.repeat(X[:, :1], m - 1, axis=1)]
    Pm = GaussNB().fit(A[tr], mot_nong(y[tr])).xac_suat(A[te])
    print(f"  x1 sao chép {m:>2} bản: độ chính xác {np.mean(Pm.argmax(1) == y[te]):.3f}, độ tin cậy TB {Pm.max(1).mean():.3f}, log-loss {log_loss(y[te], Pm):.3f}")

# Ví dụ tính tay (mục 5.7): Multinomial NB; cột = [thưởng, miễn_phí, họp, bài]; lớp 1 = rác
Xt = np.array([[2,1,0,0],[1,2,0,1],[0,0,1,2],[1,0,1,1],[0,0,2,1.]]); Yt = mot_nong(np.array([1,1,0,0,0])); x0 = np.array([[1,1,1,0.]])
m1 = MultiNB(1.0).fit(Xt, Yt)
print("\nLaplace alpha=1: điểm số (bt, rác) =", m1.diem_so(x0)[0], "-> P =", m1.xac_suat(x0)[0])
with np.errstate(divide="ignore"):
    print("alpha=0 (không làm trơn): điểm số =", MultiNB(0.0).fit(Xt, Yt).diem_so(x0)[0], "-> cả hai lớp bị loại!")

Bayes ngây thơ Gauss: test 0.833, log-loss 0.402
  q=0   đặc trưng nhiễu: Bayes ngây thơ 0.833 | k-NN 0.828
  q=20  đặc trưng nhiễu: Bayes ngây thơ 0.800 | k-NN 0.711
  q=100 đặc trưng nhiễu: Bayes ngây thơ 0.767 | k-NN 0.617
  x1 sao chép  1 bản: độ chính xác 0.833, độ tin cậy TB 0.816, log-loss 0.402
  x1 sao chép  5 bản: độ chính xác 0.761, độ tin cậy TB 0.949, log-loss 1.002
  x1 sao chép 10 bản: độ chính xác 0.744, độ tin cậy TB 0.976, log-loss 2.105

Laplace alpha=1: điểm số (bt, rác) = [-5.903 -5.337] -> P = [0.362 0.638]
alpha=0 (không làm trơn): điểm số = [-inf -inf] -> cả hai lớp bị loại!


## Mục 6. Thiên lệch–phương sai, lời nguyền chiều, nhất quán

Với bộ làm trơn tuyến tính $\hat f(x)=\sum_i s_i(x)y_i$ và thiết kế cố định, hai thành phần của sai số có **công thức đóng**:
$$\text{thiên lệch}=\sum_i s_i f(x_i)-f(x),\qquad \text{phương sai}=\sigma^2\lVert\mathbf s(x)\rVert^2 .$$
Với k-NN, $s_i=1/k$ nên phương sai $=\sigma^2/k$ (ô dưới kiểm chứng). Tổng sai số $=\sigma^2+\text{thiên lệch}^2+\text{phương sai}$ có hình chữ "U" theo $k$ hoặc $h$.

In [10]:
def phan_ra_sai_so(S, f_tren_x, f_tren_luoi, sigma):
    """Thiên lệch^2 và phương sai trung bình của bộ làm trơn y_hat = S y (thiết kế cố định).

    GIẢI THÍCH: với y = f(x) + nhiễu độc lập, E[y_hat] = S f nên thiên lệch = S f - f;
    Var[y_hat_j] = sigma^2 * sum_i S_ji^2 (mỗi y_i đóng góp S_ji * nhiễu_i, độc lập nhau).
    Với k-NN, S_ji = 1/k trên k mẫu nên phương sai = sigma^2 / k đúng như mục 6.1.
    """
    bias2 = np.mean((S @ f_tren_x - f_tren_luoi) ** 2)
    var = sigma ** 2 * np.mean(np.sum(S ** 2, axis=1))
    return float(bias2), float(var)

def ma_tran_trong_so_tai_luoi(x, luoi, tham_so, kieu):
    """Ma trận S_luoi (m x n): hàng j là trọng số hiệu dụng tại điểm lưới j."""
    D = np.abs(luoi[:, None] - x[None, :])
    if kieu == "knn":
        k = tham_so
        S = np.zeros_like(D)
        idx = np.argsort(D, axis=1, kind="stable")[:, :k]
        np.put_along_axis(S, idx, 1.0 / k, axis=1)
        return S
    G = ham_nhan(D / tham_so)
    return G / G.sum(axis=1, keepdims=True)

def ty_le_tap_trung(d, n=1000, seed=0):
    """(d_xa_nhat - d_gan_nhat)/d_gan_nhat với n điểm đều trong [0,1]^d (mục 4.6, 6.3)."""
    rng = np.random.default_rng(seed)
    P, q = rng.random((n, d)), rng.random((1, d))
    dist = np.linalg.norm(P - q, axis=1)
    return float((dist.max() - dist.min()) / dist.min())


sigma = 0.3; x, _ = tao_du_lieu_sin(100, sigma, 0); luoi = np.linspace(0, 10, 201)
print(f"{'mô hình':<12}{'thiên lệch^2':>14}{'phương sai':>12}{'tổng (+0,09)':>14}")
for ten, kieu, ts in [("k-NN k=1","knn",1),("k-NN k=6","knn",6),("k-NN k=20","knn",20),("k-NN k=50","knn",50),
                      ("NW h=0,05","nw",0.05),("NW h=0,3","nw",0.3),("NW h=0,8","nw",0.8),("NW h=2","nw",2.0)]:
    b2, v = phan_ra_sai_so(ma_tran_trong_so_tai_luoi(x, luoi, ts, kieu), ham_that_sin(x), ham_that_sin(luoi), sigma)
    print(f"{ten:<12}{b2:>14.4f}{v:>12.4f}{sigma**2 + b2 + v:>14.4f}")
print("phương sai k-NN k=6 theo sigma^2/k:", round(sigma**2 / 6, 4))

print("\nLời nguyền chiều: (xa nhất - gần nhất)/gần nhất với 1000 điểm đều trong [0,1]^d")
print({d: round(ty_le_tap_trung(d), 3) for d in (2, 10, 100, 1000)})
print("cạnh hình lập phương con chứa 1% dữ liệu:", {d: round(0.01 ** (1 / d), 3) for d in (1, 2, 10, 50, 100)})

# tính nhất quán: hai Gauss một chiều N(0,1) và N(2,1); sai số Bayes = Phi(-1) = 0,1587
rng = np.random.default_rng(0)
def lay(n):
    yy = rng.integers(0, 2, n); return rng.normal(2.0 * yy, 1.0)[:, None], yy
Xt_, yt_ = lay(4000)
for n in (10, 100, 1000, 3000):
    X_, y_ = lay(n); ks = max(1, int(np.sqrt(n))) | 1
    print(f"n={n:<5} 1-NN {np.mean(knn_du_doan(Xt_, X_, y_, 1) != yt_):.3f} | k={ks:<3} {np.mean(knn_du_doan(Xt_, X_, y_, ks) != yt_):.3f}   (Bayes 0,1587; 1-NN tiệm cận 0,2248)")

mô hình       thiên lệch^2  phương sai  tổng (+0,09)
k-NN k=1            0.0022      0.0900        0.1822
k-NN k=6            0.0038      0.0150        0.1088
k-NN k=20           0.0414      0.0045        0.1359
k-NN k=50           0.4078      0.0018        0.4996
NW h=0,05           0.0015      0.0504        0.1418
NW h=0,3            0.0036      0.0092        0.1029
NW h=0,8            0.0509      0.0037        0.1445
NW h=2              0.3370      0.0017        0.4287
phương sai k-NN k=6 theo sigma^2/k: 0.015

Lời nguyền chiều: (xa nhất - gần nhất)/gần nhất với 1000 điểm đều trong [0,1]^d
{2: 134.618, 10: 3.47, 100: 0.407, 1000: 0.119}
cạnh hình lập phương con chứa 1% dữ liệu: {1: 0.01, 2: 0.1, 10: 0.631, 50: 0.912, 100: 0.955}
n=10    1-NN 0.283 | k=3   0.235   (Bayes 0,1587; 1-NN tiệm cận 0,2248)
n=100   1-NN 0.240 | k=11  0.154   (Bayes 0,1587; 1-NN tiệm cận 0,2248)
n=1000  1-NN 0.234 | k=31  0.153   (Bayes 0,1587; 1-NN tiệm cận 0,2248)
n=3000  1-NN 0.229 | k=55  0.148   (Bayes 

## Mục 7. Đánh giá và chữ số viết tay

- **Ma trận nhầm lẫn** $\mathbf M=\mathbf Y^\top\hat{\mathbf Y}$ ($M_{ab}$: thật $a$, dự đoán $b$). $\mathrm P_c=M_{cc}/\sum_a M_{ac}$ (tổng **cột**), $\mathrm R_c=M_{cc}/\sum_b M_{cb}$ (tổng **hàng**), $\mathrm{F1}=\frac{2\mathrm P\mathrm R}{\mathrm P+\mathrm R}$.
- **Log-loss** $=-\frac1n\sum_i\ln\hat p_{i,y_i}$ phạt nặng "tự tin mà sai"; với k-NN phải làm trơn $\tilde p_c=\frac{v_c+\alpha}{k+K\alpha}$ để tránh $\ln 0$.

In [11]:
def ma_tran_nham_lan(y_that, y_du_doan, K):
    """M = Y^T Y_hat: M[a, b] = số mẫu thật là a nhưng bị dự đoán là b."""
    return mot_nong(y_that, K).T @ mot_nong(y_du_doan, K)

def precision_recall_f1(M):
    """Theo từng lớp. Precision = M_cc / tổng CỘT c; Recall = M_cc / tổng HÀNG c."""
    tp = np.diag(M).astype(float)
    p = tp / np.maximum(M.sum(axis=0), 1)
    r = tp / np.maximum(M.sum(axis=1), 1)
    f1 = 2 * p * r / np.maximum(p + r, 1e-12)
    return p, r, f1


def log_loss_lam_tron(y, P, alpha=0.0, k=None, K=None):
    if alpha > 0: P = (P * k + alpha) / (k + K * alpha)        # P là tỉ lệ phiếu v_c / k
    P = np.clip(P, 1e-15, 1.0); return float(-np.mean(np.log(P[np.arange(len(y)), y])))

X, y = tao_du_lieu_sinh_vien(200, 0); tr, te = chia_phan_tang(y, 0.3, 0)
cn = ChuanHoa().fit(X[tr]); Ztr, Zte = cn.transform(X[tr]), cn.transform(X[te])
P = knn_xac_suat(Zte, Ztr, mot_nong(y[tr]), 18); Mx = ma_tran_nham_lan(y[te], P.argmax(1), 3)
p, r, f1 = precision_recall_f1(Mx)
print("M (hàng: thật; cột: dự đoán)\n", Mx.astype(int))
print("precision", p, "| recall", r, "| F1", f1, f"| macro-F1 {f1.mean():.3f}")
print("Bẫy độ chính xác: 1000 mẫu, 50 dương, mô hình luôn đoán 'âm' -> accuracy 0,95 nhưng recall 0, F1 0.")

M (hàng: thật; cột: dự đoán)
 [[50 10  0]
 [ 8 48  4]
 [ 0  9 51]]
precision [0.862 0.716 0.927] | recall [0.833 0.8   0.85 ] | F1 [0.847 0.756 0.887] | macro-F1 0.830
Bẫy độ chính xác: 1000 mẫu, 50 dương, mô hình luôn đoán 'âm' -> accuracy 0,95 nhưng recall 0, F1 0.


**Chữ số viết tay (cần scikit-learn).** Ô dưới: (i) chọn siêu tham số bằng `GridSearchCV` (lưới **logarit** cho tham số thang đo), (ii) đối chiếu cài đặt của ta với thư viện,
(iii) log-loss của k-NN theo hệ số làm trơn. Lưu ý: `LogisticRegression` dùng $C=1/\lambda$ (C lớn = điều chuẩn *yếu*).

In [12]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB as SkGNB, MultinomialNB as SkMNB
from sklearn.linear_model import LogisticRegression

Xd, yd = load_digits(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.3, random_state=0, stratify=yd)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
cac_mo_hinh = {
    "k-NN":    (KNeighborsClassifier(),            {"n_neighbors": list(range(1, 31))}),
    "GaussNB": (SkGNB(),                           {"var_smoothing": np.logspace(-11, 1, 25)}),
    "MultiNB": (SkMNB(),                           {"alpha": np.logspace(-4, 2, 25)}),
    "Softmax": (LogisticRegression(max_iter=3000), {"C": np.logspace(-4, 3, 25)}),
}
for ten, (mh, luoi) in cac_mo_hinh.items():
    gs = GridSearchCV(mh, luoi, cv=cv).fit(Xtr, ytr)
    tham_so = {k: float(np.round(v, 4)) for k, v in gs.best_params_.items()}
    print(f"{ten:<8} {tham_so}  CV {gs.best_score_:.4f}  test {gs.score(Xte, yte):.4f}")

# đối chiếu cài đặt từ đầu với scikit-learn
Ytr = mot_nong(ytr, 10)
print("\nk-NN k=3 trùng nhãn sklearn:", np.mean(knn_du_doan(Xte, Xtr, ytr, 3) == KNeighborsClassifier(3).fit(Xtr, ytr).predict(Xte)))
g0 = GaussNB(lam=1e-9).fit(Xtr, Ytr); sg = SkGNB().fit(Xtr, ytr)
print("GaussNB: trùng nhãn", np.mean(g0.du_doan(Xte) == sg.predict(Xte)), "| |ΔP| tối đa", f"{np.abs(g0.xac_suat(Xte) - sg.predict_proba(Xte)).max():.1e}")
m0 = MultiNB(1.0).fit(Xtr, Ytr); sm0 = SkMNB(alpha=1.0).fit(Xtr, ytr)
print("MultiNB: trùng nhãn", np.mean(m0.du_doan(Xte) == sm0.predict(Xte)), "| |ΔP| tối đa", f"{np.abs(m0.xac_suat(Xte) - sm0.predict_proba(Xte)).max():.1e}")

# log-loss của k-NN (k=3) theo hệ số làm trơn: vài điểm xác suất 0 làm log-loss 'thô' bị thổi phồng
Pk = knn_xac_suat(Xte, Xtr, Ytr, 3)
print("\nk-NN k=3, log-loss theo alpha:", {a: round(log_loss_lam_tron(yte, Pk, a, 3, 10), 3) for a in (0.0, 0.001, 0.01, 0.1)})
print("số ảnh kiểm tra có xác suất 0 cho lớp thật:", int((Pk[np.arange(len(yte)), yte] == 0).sum()))

k-NN     {'n_neighbors': 1.0}  CV 0.9841  test 0.9833
GaussNB  {'var_smoothing': 0.01}  CV 0.9268  test 0.9074
MultiNB  {'alpha': 56.2341}  CV 0.9029  test 0.8981
Softmax  {'C': 0.011}  CV 0.9682  test 0.9759

k-NN k=3 trùng nhãn sklearn: 1.0
GaussNB: trùng nhãn 1.0 | |ΔP| tối đa 2.4e-11
MultiNB: trùng nhãn 1.0 | |ΔP| tối đa 1.6e-13

k-NN k=3, log-loss theo alpha: {0.0: 0.277, 0.001: 0.083, 0.01: 0.092, 0.1: 0.3}
số ảnh kiểm tra có xác suất 0 cho lớp thật: 4


## Mục 8. Mở rộng: cây quyết định và hỗn hợp Gauss

**Cây**: tại mỗi nút chọn (đặc trưng, ngưỡng) cực tiểu độ tạp Gini có trọng số $\frac{n_L}{n}G(\mathcal R_L)+\frac{n_R}{n}G(\mathcal R_R)$, $G=1-\sum_c\hat p_c^2$.
**Hỗn hợp Gauss** $p(\mathbf x)=\sum_j\pi_j\mathcal N(\mathbf x;\boldsymbol\mu_j,\boldsymbol\Sigma_j)$ học bằng **EM**: bước E tính trách nhiệm $\gamma_{ij}$;
bước M là công thức Bayes ngây thơ với $\mathbf Y$ thay bằng $\boldsymbol\Gamma$: $\hat{\mathbf M}=\mathrm{diag}(\mathbf t)^{-1}\boldsymbol\Gamma^\top\mathbf X$, $\mathbf t=\boldsymbol\Gamma^\top\mathbf 1$.

**Giải thích mã khó 6 — `log_gauss_nhieu_chieu`.** Viết $\boldsymbol\Sigma=\mathbf L\mathbf L^\top$ (Cholesky) thì $(\mathbf x-\boldsymbol\mu)\boldsymbol\Sigma^{-1}(\mathbf x-\boldsymbol\mu)^\top=\lVert\mathbf L^{-1}(\mathbf x-\boldsymbol\mu)^\top\rVert^2$ và $\ln|\boldsymbol\Sigma|=2\sum\ln L_{ii}$:
giải hệ tam giác thay vì nghịch đảo cả ma trận. Trong `gmm_em`, ta làm việc với **log** của $\pi_j\mathcal N$ rồi dùng `logsumexp` để chuẩn hóa — cùng thủ thuật ổn định số như Softmax.

In [13]:
def gini(nhan, K):
    if len(nhan) == 0:
        return 0.0
    p = np.bincount(nhan, minlength=K) / len(nhan)
    return 1.0 - float(np.sum(p ** 2))

def cac_cach_chia_gini(X, y, K):
    """Duyệt mọi (đặc trưng j, ngưỡng t = trung điểm các giá trị liền kề); trả về danh sách
    (độ tạp có trọng số, j, t) đã sắp tăng dần. Điểm chia tốt nhất là phần tử đầu (mục 8.3)."""
    ket_qua = []
    for j in range(X.shape[1]):
        gia_tri = np.unique(X[:, j])
        for t in (gia_tri[:-1] + gia_tri[1:]) / 2:
            trai, phai = y[X[:, j] <= t], y[X[:, j] > t]
            do_tap = (len(trai) * gini(trai, K) + len(phai) * gini(phai, K)) / len(y)
            ket_qua.append((do_tap, j + 1, float(t)))
    return sorted(ket_qua)

def log_gauss_nhieu_chieu(X, mu, Sigma):
    """ln N(x; mu, Sigma) cho mọi hàng của X, tính qua phân rã Cholesky (ổn định hơn nghịch đảo)."""
    d = X.shape[1]
    L = np.linalg.cholesky(Sigma)
    # GIẢI THÍCH: Sigma = L L^T nên (x-mu) Sigma^-1 (x-mu)^T = ||L^-1 (x-mu)^T||^2, và
    # ln|Sigma| = 2 sum ln L_ii. Giải hệ tam giác thay vì nghịch đảo cả ma trận.
    z = np.linalg.solve(L, (X - mu).T)
    return -0.5 * (d * np.log(2 * pi) + 2 * np.sum(np.log(np.diag(L))) + np.sum(z ** 2, axis=0))

def gmm_em(X, J, so_vong=300, dung=1e-7, seed=0, reg=1e-6):
    """Hỗn hợp Gauss bằng EM. Bước M dùng đúng công thức của Bayes ngây thơ nhưng Y -> Gamma."""
    n, d = X.shape
    rng = np.random.default_rng(seed)
    mu = X[rng.choice(n, J, replace=False)]                   # khởi tạo tâm = J mẫu ngẫu nhiên
    Sigma = np.array([np.cov(X.T).reshape(d, d) + reg * np.eye(d)] * J)
    pi_j = np.full(J, 1.0 / J)
    ll_cu = -np.inf
    for it in range(so_vong):
        # --- Bước E: trách nhiệm Gamma (n x J) ---
        logp = np.column_stack([np.log(pi_j[j]) + log_gauss_nhieu_chieu(X, mu[j], Sigma[j])
                                for j in range(J)])
        ll = float(np.mean(logsumexp(logp)))                  # log-hợp lý trung bình
        Gamma = np.exp(logp - logsumexp(logp)[:, None])
        # --- Bước M: giống mục 5.4 với Y thay bằng Gamma ("one-hot mềm") ---
        t = Gamma.sum(axis=0)                                 # số mẫu "mềm" mỗi thành phần
        pi_j = t / n
        mu = (Gamma.T @ X) / t[:, None]                       # M = diag(t)^-1 Gamma^T X
        for j in range(J):
            Z = X - mu[j]
            Sigma[j] = (Gamma[:, j][:, None] * Z).T @ Z / t[j] + reg * np.eye(d)
        if abs(ll - ll_cu) < dung:
            break
        ll_cu = ll
    return dict(pi=pi_j, mu=mu, Sigma=Sigma, ll=ll, so_vong=it + 1)

def gmm_log_mat_do(X, mo_hinh):
    logp = np.column_stack([np.log(mo_hinh["pi"][j]) + log_gauss_nhieu_chieu(X, mo_hinh["mu"][j], mo_hinh["Sigma"][j])
                            for j in range(len(mo_hinh["pi"]))])
    return logsumexp(logp)

def bic_gmm(ll_tb, n, J, d):
    so_tham_so = (J - 1) + J * d + J * d * (d + 1) // 2
    return -2.0 * ll_tb * n + so_tham_so * np.log(n)


print("Cây quyết định trên 9 sinh viên: Gini ban đầu =", round(gini(y9, 3), 3))
for do_tap, j, t in cac_cach_chia_gini(X9, y9, 3)[:4]:
    print(f"   {do_tap:.3f}  x{j} <= {t}")
print("   (ba cách đầu HÒA nhau ở 0,333)")

x = lay_mau_hai_dinh(300, 0)[:, None]; xt = lay_mau_hai_dinh(2000, 1)[:, None]
print("\nChọn J bằng BIC (dữ liệu thật có 2 thành phần):")
for J in range(1, 5):
    mh = gmm_em(x, J, seed=0)
    print(f"  J={J}: BIC {bic_gmm(mh['ll'], len(x), J, 1):8.1f} | log-hợp lý kiểm tra {np.mean(gmm_log_mat_do(xt, mh)):.4f} | {mh['so_vong']} vòng EM")
mh = gmm_em(x, 2, seed=0); o = np.argsort(mh["mu"][:, 0])
print("EM J=2: pi", mh["pi"][o], "mu", mh["mu"][o, 0], "sigma", np.sqrt(mh["Sigma"][o, 0, 0]), "(thật: 0,6/0,4; -1,5/2; 0,7/1)")
print("KDE h=0,22 để so sánh, log-hợp lý kiểm tra:", round(float(np.mean(np.log(kde(xt, x, 0.22)))), 4))

Cây quyết định trên 9 sinh viên: Gini ban đầu = 0.667
   0.333  x1 <= 5.0
   0.333  x1 <= 7.5
   0.333  x2 <= 1.5
   0.433  x1 <= 6.5
   (ba cách đầu HÒA nhau ở 0,333)

Chọn J bằng BIC (dữ liệu thật có 2 thành phần):
  J=1: BIC   1237.6 | log-hợp lý kiểm tra -2.0772 | 3 vòng EM
  J=2: BIC   1125.0 | log-hợp lý kiểm tra -1.8487 | 42 vòng EM
  J=3: BIC   1141.1 | log-hợp lý kiểm tra -1.8493 | 192 vòng EM
  J=4: BIC   1153.6 | log-hợp lý kiểm tra -1.8530 | 300 vòng EM
EM J=2: pi [0.551 0.449] mu [-1.465  1.921] sigma [0.71  0.912] (thật: 0,6/0,4; -1,5/2; 0,7/1)
KDE h=0,22 để so sánh, log-hợp lý kiểm tra: -1.8681


## Vẽ hình (tùy chọn)

Các ô dưới vẽ ba hình minh họa. File notebook này được lưu **không kèm kết quả hình**; hãy chạy trong Jupyter để xem.

In [14]:
import matplotlib.pyplot as plt

# (1) Vùng quyết định: Softmax (tham số) vs k-NN (phi tham số) trên ba vòng tròn
X, y = tao_ba_vong_tron(200, 0); cn = ChuanHoa().fit(X); Z = cn.transform(X)
gx, gy = np.meshgrid(np.linspace(-6.5, 6.5, 250), np.linspace(-6.5, 6.5, 250)); luoi2 = np.c_[gx.ravel(), gy.ravel()]
sm = SoftmaxHoiQuy().fit(X, mot_nong(y))
fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
for a, ten, pred in [(ax[0], "Softmax (tham số)", sm.du_doan(luoi2)), (ax[1], "k-NN, k=15 (phi tham số)", knn_du_doan(cn.transform(luoi2), Z, y, 15))]:
    a.contourf(gx, gy, pred.reshape(gx.shape), levels=[-.5, .5, 1.5, 2.5], colors=["#fbd9bf", "#bfe8dc", "#d9d6ec"])
    a.scatter(X[:, 0], X[:, 1], c=y, cmap="Dark2", s=6); a.set_title(ten); a.set_aspect("equal")
plt.tight_layout(); plt.show()

In [15]:
# (2) Hồi quy địa phương và (3) KDE với ba băng thông
x, yy = tao_du_lieu_sin(100, 0.3, 0); g = np.linspace(0, 10, 300)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(x, yy, s=8, c="#999"); ax[0].plot(g, ham_that_sin(g), "--k", label="hàm thật")
ax[0].plot(g, nadaraya_watson(g[:, None], x[:, None], yy, 0.3), label="Nadaraya–Watson h=0,3")
ax[0].plot(g, loess(g[:, None], x[:, None], yy, 0.35), label="LOESS h=0,35"); ax[0].legend()
xs2 = lay_mau_hai_dinh(300, 0); g2 = np.linspace(-5, 6, 400)
ax[1].plot(g2, mat_do_that_hai_dinh(g2), "--k", label="mật độ thật")
for h in (0.05, 0.22, 1.5): ax[1].plot(g2, kde(g2[:, None], xs2[:, None], h), label=f"KDE h={h}")
ax[1].legend(); plt.tight_layout(); plt.show()

## Bài tập gợi ý

1. Thêm `weights="distance"` kiểu *Gauss* cho `knn_xac_suat` (trọng số $e^{-d^2/2h^2}$) và so sánh với trọng số $1/d^2$.
2. Với dữ liệu sin, vẽ đường **bootstrap 95%** của Nadaraya–Watson và LOESS ở hai biên; giải thích vì sao LOESS ít thiên lệch hơn nhưng dải rộng hơn.
3. Mở rộng `GaussNB` thành Bayes ngây thơ dùng **KDE** (mỗi cặp lớp–đặc trưng một KDE một chiều) và so với `GaussNB` trên dữ liệu chữ số sau PCA.
4. Cài đặt `gmm_em` với hiệp phương sai **đường chéo** và so sánh BIC/độ chính xác phân loại trên chữ số (chiếu PCA 20 chiều).